In [ ]:
import json
import os
from pathlib import Path

workspace = Path(os.getenv("GITHUB_WORKSPACE") or Path.cwd()).resolve()
original_path = workspace / "Part_4.ipynb"
if not original_path.is_file():
    raise FileNotFoundError("Keep the original Part_4.ipynb beside Part_4_v2.ipynb.")

original_nb = json.loads(original_path.read_text(encoding="utf-8"))
code_cells = [cell for cell in original_nb.get("cells", []) if cell.get("cell_type") == "code"]
if len(code_cells) != 1:
    raise RuntimeError(f"Expected one code cell in original Part_4.ipynb, found {len(code_cells)}.")

raw_source = code_cells[0].get("source", "")
source = "".join(raw_source) if isinstance(raw_source, list) else str(raw_source)

def replace_once(old, new, label):
    global source
    count = source.count(old)
    if count != 1:
        raise RuntimeError(f"Safe patch stopped: expected one {label} anchor, found {count}.")
    source = source.replace(old, new, 1)

replace_once(
    '    "Generated At"\n]',
    '    "Generated At",\n'
    '    "Metadata Generation Status",\n'
    '    "Discord Upload Status",\n'
    '    "Discord Attachment URL",\n'
    '    "Discord Message URL",\n'
    '    "Discord Message ID",\n'
    '    "Discord Channel ID",\n'
    '    "Discord File Size MB",\n'
    '    "Discord Error"\n]',
    "metadata header"
)

discord_helpers = r"""
# =============================================================================
# 1A. DISCORD WEBHOOK UPLOAD — ONE VIDEO AT A TIME
# =============================================================================
import mimetypes

DISCORD_WEBHOOK_URL = os.getenv("DISCORD_WEBHOOK_URL", "").strip()
DISCORD_MAX_FILE_MB = float(os.getenv("DISCORD_MAX_FILE_MB", "20"))
DISCORD_UPLOAD_TIMEOUT_SECONDS = int(os.getenv("DISCORD_UPLOAD_TIMEOUT_SECONDS", "900"))
DISCORD_THREAD_ID = os.getenv("DISCORD_THREAD_ID", "").strip()

if not DISCORD_WEBHOOK_URL:
    raise RuntimeError("Missing DISCORD_WEBHOOK_URL. Add it as a GitHub Actions Secret.")
if DISCORD_MAX_FILE_MB <= 0:
    raise RuntimeError("DISCORD_MAX_FILE_MB must be greater than zero.")

def validate_discord_webhook():
    try:
        response = requests.get(DISCORD_WEBHOOK_URL, timeout=30)
    except requests.RequestException:
        raise RuntimeError("Could not reach the Discord webhook. Check the secret and network connection.")
    if not response.ok:
        raise RuntimeError(f"Discord webhook validation failed with HTTP {response.status_code}. Check the secret.")
    try:
        info = response.json()
    except ValueError:
        raise RuntimeError("Discord webhook validation returned an unreadable response.")
    if not info.get("channel_id"):
        raise RuntimeError("The Discord webhook did not return a channel_id. Use an incoming channel webhook.")
    return info

DISCORD_WEBHOOK_INFO = validate_discord_webhook()

def _discord_failure(status, error, size_mb):
    return {
        "status": status,
        "attachment_url": "",
        "message_url": "",
        "message_id": "",
        "channel_id": str(DISCORD_WEBHOOK_INFO.get("channel_id", "")),
        "file_size_mb": round(size_mb, 2),
        "error": str(error)[:1000],
    }

def upload_video_to_discord(video_path, video_name, metadata):
    # Called synchronously inside the video loop: one file per Discord message.
    path = Path(video_path)
    if not path.is_file():
        return _discord_failure("failed", "Video file does not exist.", 0.0)

    size_bytes = path.stat().st_size
    size_mb = size_bytes / (1024 * 1024)
    if size_bytes <= 0:
        return _discord_failure("failed", "Video file is empty.", size_mb)

    max_bytes = int(DISCORD_MAX_FILE_MB * 1024 * 1024)
    if size_bytes > max_bytes:
        return _discord_failure(
            "too_large",
            f"Video is {size_mb:.2f} MiB; configured limit is {DISCORD_MAX_FILE_MB:.2f} MiB. "
            "Increase DISCORD_MAX_FILE_MB only if your Discord account/server allows larger uploads.",
            size_mb,
        )

    title = str(metadata.get("yt_title", "") or Path(video_name).stem).strip()
    caption = f"🎬 {title}\n📁 {video_name}"
    short_caption = str(metadata.get("tiktok_caption", "") or "").strip()
    if short_caption:
        caption += "\n\n" + short_caption[:700]
    caption = caption[:1900]

    mime_type = mimetypes.guess_type(str(path))[0] or "application/octet-stream"
    params = {"wait": "true"}
    if DISCORD_THREAD_ID:
        params["thread_id"] = DISCORD_THREAD_ID

    for attempt in range(1, 6):
        try:
            with path.open("rb") as video_file:
                response = requests.post(
                    DISCORD_WEBHOOK_URL,
                    params=params,
                    data={"payload_json": json.dumps({
                        "content": caption,
                        "allowed_mentions": {"parse": []},
                    }, ensure_ascii=False)},
                    files={"files[0]": (path.name, video_file, mime_type)},
                    timeout=(30, DISCORD_UPLOAD_TIMEOUT_SECONDS),
                )
        except requests.RequestException as exc:
            if attempt == 5:
                return _discord_failure("failed", "Network error during Discord upload.", size_mb)
            time.sleep(min(2 ** (attempt - 1), 30))
            continue

        if response.status_code == 429:
            try:
                retry_after = float(response.json().get("retry_after", response.headers.get("Retry-After", 2)))
            except (ValueError, TypeError):
                retry_after = 2.0
            if attempt == 5:
                return _discord_failure("failed", "Discord rate limit persisted after five attempts.", size_mb)
            time.sleep(min(max(retry_after, 1.0), 120.0))
            continue

        if 500 <= response.status_code < 600:
            if attempt == 5:
                return _discord_failure("failed", f"Discord server error HTTP {response.status_code}.", size_mb)
            time.sleep(min(2 ** (attempt - 1), 30))
            continue

        if not response.ok:
            try:
                detail = response.json().get("message", "")
            except ValueError:
                detail = response.text[:300]
            return _discord_failure("failed", f"Discord HTTP {response.status_code}: {detail or 'upload rejected'}", size_mb)

        try:
            message = response.json()
        except ValueError:
            return _discord_failure("failed", "Discord returned no message JSON; link cannot be recorded.", size_mb)

        attachments = message.get("attachments") or []
        if not attachments or not attachments[0].get("url"):
            return _discord_failure("failed", "Discord returned no attachment URL.", size_mb)

        attachment_url = attachments[0]["url"]
        message_id = str(message.get("id", ""))
        channel_id = str(message.get("channel_id") or DISCORD_WEBHOOK_INFO.get("channel_id", ""))
        guild_id = str(DISCORD_WEBHOOK_INFO.get("guild_id") or "")
        message_url = (
            f"https://discord.com/channels/{guild_id}/{channel_id}/{message_id}"
            if guild_id and channel_id and message_id else ""
        )
        return {
            "status": "uploaded",
            "attachment_url": attachment_url,
            "message_url": message_url,
            "message_id": message_id,
            "channel_id": channel_id,
            "file_size_mb": round(size_mb, 2),
            "error": "",
        }

    return _discord_failure("failed", "Discord upload failed after five attempts.", size_mb)
"""

marker = "\n\n\n# =============================================================================\n# 2. API KEYS"
if source.count(marker) != 1:
    raise RuntimeError("Safe patch stopped: Discord helper insertion point was not unique.")
source = source.replace(marker, "\n\n\n" + discord_helpers + marker, 1)

replace_once(
    """        except Exception as exc:
            print("  Metadata generation failed completely:", exc)
            failed_count += 1
            continue""",
    """        except Exception as exc:
            print("  Metadata generation failed; continuing with the Discord video upload:", exc)
            failed_count += 1
            result = {
                "status": "metadata_failed",
                "ranking_used": False,
                "selected": normalize_metadata({}),
                "score": "",
                "candidate_scores": "",
                "model_used": "",
                "error": str(exc)[:1000],
            }
            metadata = result["selected"]""",
    "metadata failure fallback"
)

replace_once(
    '        # Local workspace only: there is no remote video URL.\n        video_url = "LOCAL_WORKSPACE_ONLY"',
    '        # Filled with the Discord attachment URL after the sequential upload.\n        video_url = ""',
    "video link placeholder"
)

replace_once(
    """        thumbnail_url = create_thumbnail(
            video_path,
            video_name
        )

        generated_at = datetime.now().strftime(""",
    """        thumbnail_url = create_thumbnail(
            video_path,
            video_name
        )

        # This upload completes before the next video starts.
        discord_result = upload_video_to_discord(video_path, video_name, metadata)
        video_url = discord_result.get("attachment_url", "")

        generated_at = datetime.now().strftime(""",
    "Discord upload call"
)

replace_once(
    """            "pending",
            formatted_time,""",
    """            discord_result.get("status", "not_attempted"),
            formatted_time,""",
    "row upload status"
)

replace_once(
    """            result.get("candidate_scores", ""),
            generated_at
        ]""",
    """            result.get("candidate_scores", ""),
            generated_at,
            result.get("status", ""),
            discord_result.get("status", "not_attempted"),
            discord_result.get("attachment_url", ""),
            discord_result.get("message_url", ""),
            discord_result.get("message_id", ""),
            discord_result.get("channel_id", ""),
            str(discord_result.get("file_size_mb", "")),
            discord_result.get("error", "")
        ]

        if len(row) != len(SHEET_HEADER):
            raise RuntimeError(
                f"Metadata row/header mismatch for {video_name}: {len(row)} values vs {len(SHEET_HEADER)} columns."
            )""",
    "metadata row Discord fields"
)

save_function = """
# Save the CSV incrementally after each video has been handled.
csv_path = LOCAL_OUTPUT_DIR / "metadata_rows.csv"

def save_metadata_rows(rows):
    with csv_path.open("w", newline="", encoding="utf-8-sig") as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow(SHEET_HEADER)
        writer.writerows(rows)

"""
main_marker = "\n\n# =============================================================================\n# 12. MAIN PROCESS"
if source.count(main_marker) != 1:
    raise RuntimeError("Safe patch stopped: main-process insertion point was not unique.")
source = source.replace(main_marker, "\n\n" + save_function + main_marker, 1)

replace_once(
    """        local_rows.append(row)
        existing_names.add(video_name)""",
    """        local_rows.append(row)
        save_metadata_rows(local_rows)
        existing_names.add(video_name)""",
    "incremental CSV save"
)

replace_once(
    """# Save a local copy of newly generated rows and a run summary.
csv_path = LOCAL_OUTPUT_DIR / "metadata_rows.csv"
with csv_path.open("w", newline="", encoding="utf-8-sig") as csv_file:
    writer = csv.writer(csv_file)
    writer.writerow(SHEET_HEADER)
    writer.writerows(local_rows)""",
    """# Final CSV flush (rows were also saved after each video).
save_metadata_rows(local_rows)

discord_status_index = SHEET_HEADER.index("Discord Upload Status")
discord_uploaded_count = sum(1 for row in local_rows if row[discord_status_index] == "uploaded")
discord_upload_failed_count = sum(1 for row in local_rows if row[discord_status_index] != "uploaded")""",
    "final CSV save"
)

replace_once(
    """    "local_thumbnail_directory": str(LOCAL_THUMBNAIL_DIR),
    "generated_at": datetime.now(SCHEDULE_TIMEZONE).isoformat(),""",
    """    "local_thumbnail_directory": str(LOCAL_THUMBNAIL_DIR),
    "discord_upload_success_count": discord_uploaded_count,
    "discord_upload_failed_count": discord_upload_failed_count,
    "discord_channel_id": str(DISCORD_WEBHOOK_INFO.get("channel_id", "")),
    "discord_file_limit_mib": DISCORD_MAX_FILE_MB,
    "generated_at": datetime.now(SCHEDULE_TIMEZONE).isoformat(),""",
    "run summary Discord counts"
)

replace_once(
    'print("Cloud uploads: disabled (local workspace only)")',
    """print("Discord uploads successful:", discord_uploaded_count)
print("Discord uploads failed or too large:", discord_upload_failed_count)
print("Discord attachment and message links saved in:", csv_path)""",
    "final Discord report"
)

compile(source, str(original_path), "exec")
print("Part_4_v2 patch checks passed; original Part_4.ipynb was not modified.")
exec(compile(source, str(original_path), "exec"), globals())
